<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# ROS 2 Nodes, Topics, and Interfaces

The previous notebook inspected the definitions of image and IMU messages. Those definitions describe how sensor data is represented, but displaying them does not establish that any data is being sent by a publisher node, or received by any subscriber node.

How can the graph reveal which nodes publish data, and how can an observer check whether their messages arrive?

A small text publisher makes both observations possible without a sensor or a custom Python program. By the end of this notebook, the distinction between a __node__, a __topic__, and a __message__ will be visible in command output. The same example will also show why communicating components need an agreed message type.

## Is anything publishing yet?

The commands in this notebook run in two terminals inside the dedicated environment described in the [README](../README.md#prerequisites). Terminal 1 will run the publisher; Terminal 2 will observe it.

Open two terminals. In Terminal 1, load the ROS 2 environment:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
```

Successful sourcing normally produces no output and returns to the prompt. It makes the installed ROS 2 commands and libraries available in that terminal.

Before starting a publisher, inspect the visible nodes:

```bash
ros2 node list
```

With no application nodes running in the dedicated environment, the command should print nothing and return to the prompt. An empty listing means that this observation discovered no visible nodes. This is the expected starting state for the activity.

If other exercises are already running, their node names may appear.

## Give the graph something to carry

A publisher needs a topic name, a message type, and values for the message fields. For example, a simple text message is sufficient for this investigation.

The familiar `ros2 interface show` command reveals the field that must be filled:

```bash
ros2 interface show std_msgs/msg/String
```

Omitting the introductory comments, this shows:

```shell
string data
```

The message contains one field, `data`, holding text. The command-line tool can create a publisher and fill that field directly.

In Terminal 1, start a publisher:

```bash
ros2 topic pub --rate 1 --node-name example_publisher \
  /chatter std_msgs/msg/String "{data: '42'}"
```

Here, `--rate 1` establishes the publishing frequency in Hertz (Hz), which corresponds to one message per second, and `--node-name` gives the publishing node an identifiable name. `/chatter` is the topic, `std_msgs/msg/String` is its message type, and `"{data: '42'}"` assigns the text `42` to the message's `data` field.

Publication output should begin with:

```shell
publisher: beginning loop
publishing #1: std_msgs.msg.String(data='42')

publishing #2: std_msgs.msg.String(data='42')
```

The numbering continues as messages are published. This command keeps running, so the terminal does not return to its prompt. Leave it running while the second terminal inspects the graph.

## Find the node and its topic

In Terminal 2, load the same ROS 2 environment:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
```

As before, successful sourcing normally produces no output. Now repeat the node listing:

```bash
ros2 node list
```

The previously empty listing should now contain the node created by the publishing command:

```shell
/example_publisher
```

The next question is which topic carries its data. The `-t` option includes message types in the topic listing:

```bash
ros2 topic list -t
```

The listing should include:

```shell
/chatter [std_msgs/msg/String]
/parameter_events [rcl_interfaces/msg/ParameterEvent]
/rosout [rcl_interfaces/msg/Log]
```

The `/chatter` entry matches the topic and type selected in Terminal 1. The other two entries support parameter-change notifications and ROS logging; they are not additional sensor streams.

The distinction is now visible: `/example_publisher` names a participating node, while `/chatter` names a message stream. The node performs the publishing work; the topic identifies where subscribers can receive its messages.


## Check that a message actually arrives

The publisher is running, and the second terminal has discovered its node and topic. Receiving a message is the next check because a visible interface alone does not establish delivery.

To "listen" to the data flowing inside topics, the `ros2 topic echo` command is used, which creates a temporary subscriber. The `--once` option requests one message and then exits. In Terminal 2, run:

```bash
ros2 topic echo --once /chatter
```

The received message should appear as:

```shell
data: '42'
---
```

The `data` field contains the text sent by Terminal 1. The `---` marks the end of the displayed message. After receiving it, the command returns to the prompt. To observe a continuous stream, omit `--once` from `ros2 topic echo`. Stop the continuous observation with `Ctrl-C` before entering the next command in Terminal 2.

With the publisher running, a message should arrive shortly. If the command remains silent, it is still waiting; that is different from the empty node listing before the publisher started. The first check is whether Terminal 1 is still publishing and both terminals were opened in the same exercise environment. `Ctrl-C` ends a waiting observation.

The publisher did not need to name this subscriber. It published on `/chatter`, and the temporary subscriber joined that topic. Other subscribers can join the same stream without changing the publisher.

## Why must the message types agree?

The displayed value resembles the number 42, but the published message contains text. Would a subscriber expecting an integer receive the same data?

The integer message definition makes the difference visible:

```bash
ros2 interface show std_msgs/msg/Int32
```

Omitting introductory comments, the definition contains:

```shell
int32 data
```

Both message definitions use the field name `data`, but one field contains text and the other contains a signed 32-bit integer. A subscriber expecting `Int32` does not receive the publisher's `String` messages merely because the topic and field names match. ROS 2 does not automatically convert the text into an integer.

The meaning of a value also belongs to the interface agreement. Two components exchanging a distance can agree on a numerical field type yet disagree about whether its value represents metres or centimetres. A usable interface defines both the structure and the meaning of its data.

Delivery policies must also be compatible. ROS 2 calls these __quality of service (QoS)__ settings. The defaults used by these commands support this exchange. Other publishers, such as sensor bridges, can use different policies, so a subscriber must use compatible settings.


## When a stream is not the whole interaction

The text exchange demonstrates the same communication pattern used for successive camera images: a publisher produces a stream, and subscribers receive its messages.

Other needs suggest the interfaces compared in [Table 1](#table-1).

<table id="table-1" class="lx-table">
  <caption>Table 1: ROS 2 interfaces for different application needs.</caption>
  <thead>
    <tr>
      <th>Need</th>
      <th>ROS 2 interface</th>
      <th>What the interface provides</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Receive successive images</td>
      <td>Topic</td>
      <td>A stream of messages</td>
    </tr>
    <tr>
      <td>Request a component's current status</td>
      <td>Service</td>
      <td>A request and a response</td>
    </tr>
    <tr>
      <td>Navigate to a goal while reporting progress</td>
      <td>Action</td>
      <td>A goal, feedback, a final result, and cancellation</td>
    </tr>
    <tr>
      <td>Configure a node's controller gain</td>
      <td>Parameter</td>
      <td>A named configuration value owned by the node</td>
    </tr>
  </tbody>
</table>

For example, a navigation task needs more than a stream of position readings. Another component must be able to submit a goal, follow progress, and request cancellation. An action expresses that interaction.

A parameter serves a different purpose: it configures a node's behavior. The node must expose and use the parameter; a Python constant does not automatically become one.

These examples illustrate interface choices. They do not imply that the corresponding navigation or configuration interfaces are running in the exercise environment.


## Finish the observation

Stop the publisher with `Ctrl-C` in Terminal 1. Its publication output stops and the shell prompt returns.

In Terminal 2, repeat:

```bash
ros2 node list
```

The listing should become empty again after discovery reflects the publisher's departure. If other application nodes are running, their names may remain.

The before-and-after observations connect the running program to its presence in the graph.

The investigation began with an installed message definition and no visible application nodes. Starting a publisher added a named node and a typed topic; receiving a message established delivery. Stopping the publisher removed that node from the graph.

These observations depended on both terminals discovering the same participants. Discovery settings determine which participants each terminal can find.

## Further reading

The ROS 2 Jazzy documentation describes [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html), [services](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Services.html), [actions](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Actions.html), and [parameters](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Parameters.html).

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
